# Chapitre 5 - Cours 1 : réplication à leader unique

Répliquer, c'est garder une copie des mêmes données sur plusieurs machines reliées par
un réseau. Tant que les données ne changent pas, c'est trivial : on copie une fois. Toute
la difficulté vient des **modifications**. Ce premier cours étudie la famille la plus
répandue, la réplication à **leader unique** (leader-based, primary-backup,
active/passive) : un nœud reçoit toutes les écritures et les transmet aux autres. Il
couvre l'introduction du chapitre et la section *Leaders and Followers* jusqu'à
*Handling Node Outages* inclus.

**Plan.**

1. Trois raisons de répliquer, et le schéma leader-suiveurs.
2. Synchrone, asynchrone, semi-synchrone : latence et disponibilité des écritures.
3. Ajouter un suiveur sans arrêter le leader.
4. Panne d'un suiveur : le rattrapage.
5. Panne du leader : le basculement et les écritures perdues.
6. Split brain, fencing et clés réutilisées.

Les simulations reposent sur le simulateur à événements `ddia.sim` et sur le module
[`ddia/replication.py`](../../ddia/replication.py) (classes `ReplNode`, `Controller`,
`WriterClient`, fonction `failover_scenario`).

In [1]:
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ddia import replication as rp
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)
from ddia.sim import Network, Node
from ddia.timelines import SpaceTime

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)